# Author: [Sunny Shaban Ali](https://github.com/sunnyallana)

# Reference: [NgoQuocBao1010](https://github.com/NgoQuocBao1010/Exercise-Correction/tree/main)

## Installing dependencies

In [13]:
!pip install keras-tuner mediapipe

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 17.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
ydf 0.12.0 requires protobuf<6.0.0,>=5.29.1, but you have protobuf 4.25.8 which is inco

## Plank detection deep learning model pipeline with 7 layers and dropout

---



In [11]:
# Plank Exercise Classification - 7 Layer Model with Dropout
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

def round_up_metric_results(results) -> list:
    """Round up metrics results such as precision score, recall score, ..."""
    return list(map(lambda el: round(el, 3), results))

def describe_model(model):
    """Describe Model architecture"""
    print(f"Model architecture:")
    for i, layer in enumerate(model.layers):
        number_of_units = layer.units if hasattr(layer, 'units') else 0
        if hasattr(layer, "activation"):
            print(f"Layer-{i + 1}: {number_of_units} units, activation: {layer.activation}")
        else:
            print(f"Layer-{i + 1}: {number_of_units} units, activation: None")

def get_best_model(tuner):
    """Describe and return the best model found from keras tuner"""
    best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
    best_model = tuner.hypermodel.build(best_hps)
    describe_model(best_model)
    print("\nBest hyperparameters:")
    ignore_params = ["tuner", "activation", "layer"]
    for param, value in best_hps.values.items():
        if not any(word in param for word in ignore_params):
            print(f"{param}: {value}")
    return best_model

def model_7lD_builder(hp):
    """7-layer model builder with dropout for hyperparameter tuning"""
    model = Sequential()
    model.add(Dense(68, input_dim=68, activation="relu"))

    # Hyperparameters
    hp_activation = hp.Choice('activation', values=['relu', 'tanh'])
    hp_layer_1 = hp.Int('layer_1', min_value=32, max_value=512, step=32)
    hp_layer_2 = hp.Int('layer_2', min_value=32, max_value=512, step=32)
    hp_layer_3 = hp.Int('layer_3', min_value=32, max_value=512, step=32)
    hp_dropout_1 = hp.Float('dropout_1', min_value=0.1, max_value=0.5, step=0.1)
    hp_dropout_2 = hp.Float('dropout_2', min_value=0.1, max_value=0.5, step=0.1)
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    # Model layers
    model.add(Dense(units=hp_layer_1, activation=hp_activation))
    model.add(Dropout(rate=hp_dropout_1))
    model.add(Dense(units=hp_layer_2, activation=hp_activation))
    model.add(Dropout(rate=hp_dropout_2))
    model.add(Dense(units=hp_layer_3, activation=hp_activation))
    model.add(Dense(3, activation="softmax"))

    model.compile(optimizer=Adam(learning_rate=hp_learning_rate),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

# Load and preprocess training data
print("Loading training data...")
df = pd.read_csv("./train.csv")

# Categorize labels
df.loc[df["label"] == "C", "label"] = 0
df.loc[df["label"] == "H", "label"] = 1
df.loc[df["label"] == "L", "label"] = 2

print(f'Training data - Rows: {df.shape[0]}, Columns: {df.shape[1]}')
print(f"Label distribution:\n{df['label'].value_counts()}\n")

# Create and save input scaler
print("Creating input scaler...")
# Create model directory if it doesn't exist
os.makedirs("./model", exist_ok=True)

# Prepare features for scaling
x_features = df.drop("label", axis=1)

# Create and fit scaler
sc = StandardScaler()
sc.fit(x_features)

# Save the scaler
with open("./model/input_scaler.pkl", "wb") as f:
    pickle.dump(sc, f)
print("Input scaler saved to ./model/input_scaler.pkl")

# Apply scaling to features
x = pd.DataFrame(sc.transform(x_features))
y = df["label"]
y_cat = to_categorical(y)

# Train-test split
x_train, x_test, y_train, y_test = train_test_split(x.values, y_cat, test_size=0.2, random_state=1234)

# Early stopping callback
stop_early = EarlyStopping(monitor='loss', patience=3)

# Hyperparameter tuning
print("Starting hyperparameter tuning...")
tuner_7lD = kt.Hyperband(
    model_7lD_builder,
    objective='accuracy',
    max_epochs=10,
    directory='keras_tuner_dir',
    project_name='plank_7layer_dropout'
)

tuner_7lD.search(x_train, y_train, epochs=10, callbacks=[stop_early])

# Get best model and train
print("Training best model...")
model_7lD = get_best_model(tuner_7lD)
history = model_7lD.fit(x_train, y_train,
                        epochs=100,
                        batch_size=10,
                        validation_data=(x_test, y_test),
                        callbacks=[stop_early])

# Evaluate on test set
print("\nEvaluating on test set...")
test_df = pd.read_csv("./test.csv")

# Categorize test labels
test_df.loc[test_df["label"] == "C", "label"] = 0
test_df.loc[test_df["label"] == "H", "label"] = 1
test_df.loc[test_df["label"] == "L", "label"] = 2

print(f'Test data - Rows: {test_df.shape[0]}, Columns: {test_df.shape[1]}')
print(f"Test label distribution:\n{test_df['label'].value_counts()}\n")

# Preprocess test data
test_x = test_df.drop("label", axis=1)
test_x = pd.DataFrame(sc.transform(test_x))
test_y = test_df["label"]
test_y_cat = to_categorical(test_y)

# Make predictions
predict_x = model_7lD.predict(test_x, verbose=False)
y_pred_class = np.argmax(predict_x, axis=1)
y_test_class = np.argmax(test_y_cat, axis=1)

# Calculate metrics
cm = confusion_matrix(y_test_class, y_pred_class, labels=[0, 1, 2])
(p_score, r_score, f_score, _) = precision_recall_fscore_support(y_test_class, y_pred_class, labels=[0, 1, 2])

print("Test Results:")
print(f"Precision Score: {round_up_metric_results(p_score)}")
print(f"Recall Score: {round_up_metric_results(r_score)}")
print(f"F1 Score: {round_up_metric_results(f_score)}")
print(f"Confusion Matrix:\n{cm}")

# Save the model
print("\nSaving model...")
with open("./model/plank_7layer_dropout.pkl", "wb") as f:
    pickle.dump(model_7lD, f)

print("Model saved successfully as 'plank_7layer_dropout.pkl'!")

Trial 30 Complete [00h 00m 56s]
accuracy: 0.9990795850753784

Best accuracy So Far: 0.9993863701820374
Total elapsed time: 00h 13m 48s
Training best model...
Model architecture:
Layer-1: 68 units, activation: <function relu at 0x7d034d833b00>
Layer-2: 352 units, activation: <function relu at 0x7d034d833b00>
Layer-3: 0 units, activation: None
Layer-4: 448 units, activation: <function relu at 0x7d034d833b00>
Layer-5: 0 units, activation: None
Layer-6: 512 units, activation: <function relu at 0x7d034d833b00>
Layer-7: 3 units, activation: <function softmax at 0x7d02ddb744a0>

Best hyperparameters:
dropout_1: 0.1
dropout_2: 0.30000000000000004
learning_rate: 0.0001
Epoch 1/100
2282/2282 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - accuracy: 0.9095 - loss: 0.2498 - val_accuracy: 0.9984 - val_loss: 0.0043
Epoch 2/100
2282/2282 ━━━━━━━━━━━━━━━━━━━━ 19s 8ms/step - accuracy: 0.9973 - loss: 0.0111 - val_accuracy: 0.9993 - val_loss: 0.0025
Epoch 3/100
2282/2282 ━━━━━━━━━━━━━━━━━━━━ 22s 10ms/step - accuracy

## Google Colab Video Processing

In [14]:
import mediapipe as mp
import cv2
import numpy as np
import pandas as pd
import pickle
import warnings
from IPython.display import HTML
from base64 import b64encode
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import display, clear_output
import time

warnings.filterwarnings('ignore')

# Drawing helpers
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Determine important landmarks for plank
IMPORTANT_LMS = [
    "NOSE",
    "LEFT_SHOULDER",
    "RIGHT_SHOULDER",
    "LEFT_ELBOW",
    "RIGHT_ELBOW",
    "LEFT_WRIST",
    "RIGHT_WRIST",
    "LEFT_HIP",
    "RIGHT_HIP",
    "LEFT_KNEE",
    "RIGHT_KNEE",
    "LEFT_ANKLE",
    "RIGHT_ANKLE",
    "LEFT_HEEL",
    "RIGHT_HEEL",
    "LEFT_FOOT_INDEX",
    "RIGHT_FOOT_INDEX",
]

# Generate all columns of the data frame
HEADERS = ["label"]  # Label column
for lm in IMPORTANT_LMS:
    HEADERS += [f"{lm.lower()}_x", f"{lm.lower()}_y", f"{lm.lower()}_z", f"{lm.lower()}_v"]

def extract_important_keypoints(results) -> list:
    """Extract important keypoints from mediapipe pose detection"""
    landmarks = results.pose_landmarks.landmark
    data = []
    for lm in IMPORTANT_LMS:
        keypoint = landmarks[mp_pose.PoseLandmark[lm].value]
        data.append([keypoint.x, keypoint.y, keypoint.z, keypoint.visibility])
    return np.array(data).flatten().tolist()

def rescale_frame(frame, percent=50):
    """Rescale a frame to a certain percentage compare to its original frame"""
    width = int(frame.shape[1] * percent / 100)
    height = int(frame.shape[0] * percent / 100)
    dim = (width, height)
    return cv2.resize(frame, dim, interpolation=cv2.INTER_AREA)

def process_video_colab(video_path, model_path="./model/plank_7layer_dropout.pkl",
                       scaler_path="./model/input_scaler.pkl", output_path="output_video.mp4"):
    """Process video for plank detection in Google Colab"""

    # Load model and scaler
    print("Loading model and scaler...")
    with open(model_path, "rb") as f:
        deep_learning_model = pickle.load(f)

    with open(scaler_path, "rb") as f:
        input_scaler = pickle.load(f)

    print("Models loaded successfully!")

    # Open video
    cap = cv2.VideoCapture(video_path)

    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Video properties: {width}x{height} @ {fps}fps, {total_frames} frames")

    # Setup video writer for output
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width//2, height//2))

    current_stage = ""
    prediction_probability_threshold = 0.6
    frame_count = 0

    with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        while cap.isOpened():
            ret, image = cap.read()
            if not ret:
                break

            frame_count += 1
            if frame_count % 30 == 0:  # Progress update every 30 frames
                print(f"Processing frame {frame_count}/{total_frames} ({frame_count/total_frames*100:.1f}%)")

            # Reduce size of a frame
            image = rescale_frame(image, 50)

            # Recolor image from BGR to RGB for mediapipe
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image_rgb.flags.writeable = False
            results = pose.process(image_rgb)

            if not results.pose_landmarks:
                # If no pose detected, just write the frame
                cv2.putText(image, "NO POSE DETECTED", (50, 50),
                           cv2.FONT_HERSHEY_COMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                out.write(image)
                continue

            # Draw landmarks and connections
            mp_drawing.draw_landmarks(
                image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(244, 117, 66), thickness=2, circle_radius=2),
                mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=1)
            )

            # Make detection
            try:
                # Extract keypoints from frame for the input
                row = extract_important_keypoints(results)
                X = pd.DataFrame([row], columns=HEADERS[1:])
                X = pd.DataFrame(input_scaler.transform(X))

                # Make prediction and its probability
                prediction = deep_learning_model.predict(X, verbose=0)
                predicted_class = np.argmax(prediction, axis=1)[0]
                prediction_probability = max(prediction.tolist()[0])

                # Evaluate model prediction
                if predicted_class == 0 and prediction_probability >= prediction_probability_threshold:
                    current_stage = "Correct"
                elif predicted_class == 2 and prediction_probability >= prediction_probability_threshold:
                    current_stage = "Low back"
                elif predicted_class == 1 and prediction_probability >= prediction_probability_threshold:
                    current_stage = "High back"
                else:
                    current_stage = "Unknown"

                # Visualization
                # Status box
                cv2.rectangle(image, (0, 0), (550, 60), (245, 117, 16), -1)

                # Display detection
                cv2.putText(image, "DETECTION", (95, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
                cv2.putText(image, current_stage, (90, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

                # Display class
                cv2.putText(image, "CLASS", (350, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
                cv2.putText(image, str(predicted_class), (345, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

                # Display probability
                cv2.putText(image, "PROB", (15, 12), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
                cv2.putText(image, str(round(prediction_probability, 2)), (10, 40), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            except Exception as e:
                print(f"Error processing frame {frame_count}: {e}")
                cv2.putText(image, "PROCESSING ERROR", (50, 50),
                           cv2.FONT_HERSHEY_COMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)

            # Write frame to output video
            out.write(image)

    # Release everything
    cap.release()
    out.release()

    print(f"Video processing complete! Output saved to: {output_path}")
    return output_path

def display_video_colab(video_path):
    """Display video in Google Colab"""
    from IPython.display import HTML
    from base64 import b64encode

    video_file = open(video_path, "r+b").read()
    video_url = f"data:video/mp4;base64,{b64encode(video_file).decode()}"
    return HTML(f'<video width="600" controls><source src="{video_url}"></video>')

# Example usage in Google Colab:
if __name__ == "__main__":
    # Upload your video file to Colab first, then specify the path
    VIDEO_PATH = "plank_demo.mp4"  # Change this to your uploaded video path

    print("Starting plank detection on video...")
    output_video = process_video_colab(VIDEO_PATH)

    # Display the processed video
    print("Displaying processed video:")
    display(display_video_colab(output_video))

Starting plank detection on video...
Loading model and scaler...
Models loaded successfully!
Video properties: 1920x1080 @ 30fps, 460 frames
Processing frame 30/460 (6.5%)
Processing frame 60/460 (13.0%)
Processing frame 90/460 (19.6%)
Processing frame 120/460 (26.1%)
Processing frame 150/460 (32.6%)
Processing frame 180/460 (39.1%)
Processing frame 210/460 (45.7%)
Processing frame 240/460 (52.2%)
Processing frame 270/460 (58.7%)
Processing frame 300/460 (65.2%)
Processing frame 330/460 (71.7%)
Processing frame 360/460 (78.3%)
Processing frame 390/460 (84.8%)
Processing frame 420/460 (91.3%)
Processing frame 450/460 (97.8%)
Video processing complete! Output saved to: output_video.mp4
Displaying processed video:


## Laptop Camera Real-time

In [15]:
import mediapipe as mp
import cv2
import numpy as np
import pandas as pd
import pickle
import warnings
import time

warnings.filterwarnings('ignore')

# Drawing helpers
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Determine important landmarks for plank
IMPORTANT_LMS = [
    "NOSE",
    "LEFT_SHOULDER",
    "RIGHT_SHOULDER",
    "LEFT_ELBOW",
    "RIGHT_ELBOW",
    "LEFT_WRIST",
    "RIGHT_WRIST",
    "LEFT_HIP",
    "RIGHT_HIP",
    "LEFT_KNEE",
    "RIGHT_KNEE",
    "LEFT_ANKLE",
    "RIGHT_ANKLE",
    "LEFT_HEEL",
    "RIGHT_HEEL",
    "LEFT_FOOT_INDEX",
    "RIGHT_FOOT_INDEX",
]

# Generate all columns of the data frame
HEADERS = ["label"]  # Label column
for lm in IMPORTANT_LMS:
    HEADERS += [f"{lm.lower()}_x", f"{lm.lower()}_y", f"{lm.lower()}_z", f"{lm.lower()}_v"]

def extract_important_keypoints(results) -> list:
    """Extract important keypoints from mediapipe pose detection"""
    landmarks = results.pose_landmarks.landmark
    data = []
    for lm in IMPORTANT_LMS:
        keypoint = landmarks[mp_pose.PoseLandmark[lm].value]
        data.append([keypoint.x, keypoint.y, keypoint.z, keypoint.visibility])
    return np.array(data).flatten().tolist()

def rescale_frame(frame, percent=50):
    """Rescale a frame to a certain percentage compare to its original frame"""
    width = int(frame.shape[1] * percent / 100)
    height = int(frame.shape[0] * percent / 100)
    dim = (width, height)
    return cv2.resize(frame, dim, interpolation=cv2.INTER_AREA)

def run_plank_detection_camera(model_path="./model/plank_7layer_dropout.pkl",
                              scaler_path="./model/input_scaler.pkl",
                              camera_index=0):
    """Run real-time plank detection using laptop camera"""

    print("Loading model and scaler...")
    try:
        # Load model and scaler
        with open(model_path, "rb") as f:
            deep_learning_model = pickle.load(f)

        with open(scaler_path, "rb") as f:
            input_scaler = pickle.load(f)

        print("Models loaded successfully!")
    except FileNotFoundError as e:
        print(f"Error loading models: {e}")
        print("Make sure you have trained the model and saved the scaler first!")
        return

    # Initialize camera
    print(f"Initializing camera (index: {camera_index})...")
    cap = cv2.VideoCapture(camera_index)

    if not cap.isOpened():
        print("Error: Could not open camera. Try different camera index (0, 1, 2, etc.)")
        return

    # Set camera properties (optional)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    print("Camera initialized successfully!")
    print("Instructions:")
    print("- Position yourself in front of the camera")
    print("- Get into plank position")
    print("- Press 'q' to quit")
    print("- Press 's' to save current frame")

    current_stage = ""
    prediction_probability_threshold = 0.6
    frame_count = 0
    fps_counter = 0
    start_time = time.time()

    with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        while cap.isOpened():
            ret, image = cap.read()
            if not ret:
                print("Failed to read from camera")
                break

            frame_count += 1
            fps_counter += 1

            # Calculate FPS every second
            current_time = time.time()
            if current_time - start_time >= 1.0:
                fps = fps_counter / (current_time - start_time)
                fps_counter = 0
                start_time = current_time
            else:
                fps = 0

            # Flip image horizontally for mirror effect
            image = cv2.flip(image, 1)

            # Reduce size of frame for better performance
            image = rescale_frame(image, 70)  # 70% of original size

            # Recolor image from BGR to RGB for mediapipe
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image_rgb.flags.writeable = False
            results = pose.process(image_rgb)

            # Recolor back to BGR for OpenCV
            image_rgb.flags.writeable = True
            image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

            if not results.pose_landmarks:
                # Display message when no pose is detected
                cv2.putText(image, "NO POSE DETECTED", (50, 50),
                           cv2.FONT_HERSHEY_COMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                cv2.putText(image, "Stand in front of camera", (50, 90),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2, cv2.LINE_AA)
            else:
                # Draw landmarks and connections
                mp_drawing.draw_landmarks(
                    image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(244, 117, 66), thickness=2, circle_radius=2),
                    mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=1)
                )

                # Make detection
                try:
                    # Extract keypoints from frame for the input
                    row = extract_important_keypoints(results)
                    X = pd.DataFrame([row], columns=HEADERS[1:])
                    X = pd.DataFrame(input_scaler.transform(X))

                    # Make prediction and its probability
                    prediction = deep_learning_model.predict(X, verbose=0)
                    predicted_class = np.argmax(prediction, axis=1)[0]
                    prediction_probability = max(prediction.tolist()[0])

                    # Evaluate model prediction
                    if predicted_class == 0 and prediction_probability >= prediction_probability_threshold:
                        current_stage = "Correct"
                        stage_color = (0, 255, 0)  # Green
                    elif predicted_class == 2 and prediction_probability >= prediction_probability_threshold:
                        current_stage = "Low back"
                        stage_color = (0, 0, 255)  # Red
                    elif predicted_class == 1 and prediction_probability >= prediction_probability_threshold:
                        current_stage = "High back"
                        stage_color = (0, 165, 255)  # Orange
                    else:
                        current_stage = "Unknown"
                        stage_color = (128, 128, 128)  # Gray

                    # Get image dimensions for positioning
                    h, w = image.shape[:2]

                    # Status box
                    cv2.rectangle(image, (0, 0), (w, 80), (245, 117, 16), -1)

                    # Display detection
                    cv2.putText(image, "PLANK DETECTION", (10, 20),
                               cv2.FONT_HERSHEY_COMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
                    cv2.putText(image, current_stage, (10, 50),
                               cv2.FONT_HERSHEY_COMPLEX, 1.2, stage_color, 2, cv2.LINE_AA)

                    # Display class and probability on the right
                    cv2.putText(image, f"Class: {predicted_class}", (w-200, 25),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
                    cv2.putText(image, f"Prob: {prediction_probability:.2f}", (w-200, 50),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)

                except Exception as e:
                    cv2.putText(image, f"Error: {str(e)[:30]}", (10, 100),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)

            # Display FPS
            if fps > 0:
                cv2.putText(image, f"FPS: {fps:.1f}", (10, image.shape[0] - 10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

            # Display instructions
            cv2.putText(image, "Press 'q' to quit, 's' to save frame",
                       (10, image.shape[0] - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

            # Show the image
            cv2.imshow("Plank Detection - Real-time", image)

            # Handle key presses
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                print("Quitting...")
                break
            elif key == ord('s'):
                # Save current frame
                filename = f"plank_frame_{frame_count}.jpg"
                cv2.imwrite(filename, image)
                print(f"Frame saved as {filename}")

    # Clean up
    cap.release()
    cv2.destroyAllWindows()

    # Fix for macOS window closing issue
    for i in range(1, 5):
        cv2.waitKey(1)

    print("Camera session ended.")

# Example usage
if __name__ == "__main__":
    print("Starting real-time plank detection...")

    # Try different camera indices if default doesn't work
    # Common indices: 0 (built-in), 1 (external USB camera)
    run_plank_detection_camera(camera_index=0)

Starting real-time plank detection...
Loading model and scaler...
Models loaded successfully!
Initializing camera (index: 0)...
Error: Could not open camera. Try different camera index (0, 1, 2, etc.)
